## Visualize the visits overlapping a single point on the sky



In [ ]:
# Some modules you're likely to want .. add whatever is needed.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp

In [ ]:
from rubin_sim.data import get_baseline

In [ ]:
# Import MAF
import rubin_sim.maf as maf

# import rubin_sim.utils as rsUtils
import rubin_scheduler.utils as rsUtils

Use the current baseline simulation included with `$RUBIN_SIM_DATA_DIR`

## 1. Configuration

In [ ]:
opsdb_fname = get_baseline()
run_name = os.path.split(opsdb_fname)[-1].replace(".db", "")
print(f"Using {run_name}, to be read from {opsdb_fname}")

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="01_singlevisitpoint_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")
resultsDb = maf.db.ResultsDb(data_dir)

In [ ]:
ddf_coords = {
    "COSMOS": {"ra": 150.11, "dec": 2.23},
    "XMM-LSS": {"ra": 35.57, "dec": -4.82},
    "ECDFS": {  # Extended Chandra Deep Field South
        "ra": 52.98,
        "dec": -28.12,
    },
    "ELAIS-S1": {"ra": 9.45, "dec": -44.02},
    "EDFS_a": {  # Euclid Deep Field South (split en deux pointings)
        "ra": 58.90,
        "dec": -49.32,
    },
    "EDFS_b": {"ra": 63.60, "dec": -47.60},
}

DDF_NAME = "COSMOS"

## 2. Slicer

And let's set up a slicer that will give us the observations that overlap a single point on the sky.

In [ ]:
# Specify ra / dec of the point we want to work with on the sky - in degrees.
# (these can be lists, if you want to work on multiple, limited points on the sky)

test_ra = ddf_coords[DDF_NAME]["ra"]
test_dec = ddf_coords[DDF_NAME]["dec"]

test_slicer = maf.UserPointsSlicer(test_ra, test_dec)

Using the 'Pass' metric will just return these columns from the database. 

## 3. Set up the metric

In [ ]:
# Set up the metric
cols = ["observationStartMJD", "fieldRA", "fieldDec", "rotTelPos", "rotSkyPos", "band", "night"]
mymetric = maf.PassMetric(cols=cols)

In [ ]:
# Define a sqlconstraint, if we need to just use a (large) subset of the opsim visits
# sqlconstraint = None   # no constraint, make all visits available

# Examples of other potentially useful sqlconstraints:
sqlconstraint = 'band = "r"'  # just select the visits in a particular filter
# sqlconstraint = 'scheduler_note not like "%DD%"'  # don't choose any of the DD field visits
# sqlconstraint = 'night < 365'  # only use visits in the first year of the survey

In [ ]:
# We already defined the slicer - combine the metric, slicer and sqlconstraint in a MetricBundle:
bundle = maf.MetricBundle(mymetric, test_slicer, sqlconstraint, run_name=run_name)

## 4. Run the metrics

In [ ]:
# Pass the bundle (along with any other bundles to be run on this opsim) to a MetricBundleGroup in order to
# calculate the metric bundle values.
g = maf.MetricBundleGroup({"test_metric": bundle}, opsdb_fname, out_dir=data_dir, results_db=resultsDb)
# And calculate the metric
g.run_all()

And then you can look at the `bundle.metric_values` to see what your metric calculated and how well things worked.

In [ ]:
bundle.metric_values[0][0:2]

In [ ]:
bundle.metric_values.compressed()

In [ ]:
# g.simData is the simulation visit data that the previous MetricBundleGroup queried from the database
# -- so in this case, because of the "Pass" metric, it's just another way to access the same information as 'bundle.metricValues'
g.sim_data[0:2]

## 5. Plot

So let's try to visualize the visits

In [ ]:
visits = bundle.metric_values[0]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
# How far is the center of each visit from the point on the sky? (radius is in degrees)
radius = rsUtils.angular_separation(visits["fieldRA"], visits["fieldDec"], test_ra, test_dec)
n, b, p = ax.hist(radius, bins=50)
ax.set_xlabel("Distance from center of FOV (deg)")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
ax1, ax2 = axs.flatten()

# what is the camera rotation angle (should run -90 to +90 but scheduler outputs map into 0-360)
rotAngle = visits["rotTelPos"]
n, b, p = ax1.hist(rotAngle, bins=np.arange(-100, 370, 10), alpha=0.4)
rotAngle = np.where(visits["rotTelPos"] > 180, visits["rotTelPos"] % 180 - 180, visits["rotTelPos"])
n, b, p = ax2.hist(rotAngle, bins=b, alpha=0.4)
plt.suptitle("rotTelPos")
plt.show()

In [ ]:
plt.figure()
n, b, p = plt.hist(rotAngle, bins=50)
plt.xlabel("Rotator angle (deg)")
plt.show()

In [ ]:
# Angle between "north" and the top of the FOV
n, b, p = plt.hist(visits["rotSkyPos"], bins=50)
plt.xlabel("Rot Sky Pos (deg)")

In [ ]:
x, y = rsUtils.gnomonic_project_toxy(
    np.radians(visits["fieldRA"]), np.radians(visits["fieldDec"]), np.radians(test_ra), np.radians(test_dec)
)
x = np.degrees(x)
y = np.degrees(y)
rad = np.sqrt(x**2 + y**2)

fig, axs = plt.subplots(1, 2, figsize=(16, 6))
ax1, ax2 = axs.flatten()
n, b, p = ax1.hist(rad, bins=50, alpha=0.3, facecolor="b")
n, b, p = ax2.hist(radius, bins=50, alpha=0.3, facecolor="r")
ax1.set_xlabel("rad (deg)")
ax2.set_xlabel("radius (deg)")
plt.show()

In [ ]:
show = np.where((visits["band"] == "r") & (visits["night"] < 365 * 0.5))
print(len(show[0]))

fig, ax = plt.subplots()
ax.axis("equal")
# These are just the center of the pointings - replace with patches
# plt.plot(x, y, 'k.')
# Might be able to do a hack?
# plt.scatter([0], [0], s=10000,
#            #marker='o', markersize=100, linestyle=''
#            edgecolor='black', facecolor='red', alpha=0.2)
plt.scatter(
    x[show],
    y[show],
    s=17000,
    # marker='o', markersize=100, linestyle=''
    edgecolor="black",
    facecolor="orange",
    alpha=0.1,
)
plt.plot(x[show], y[show], marker=".", markersize=5, linestyle="", color="gray")
plt.plot([0], [0], marker="x", markersize=8, linestyle="", color="k")
plt.xlim(-5, 5)
plt.ylim(-5, 5)
plt.show()

Let's take a slightly different approach. What if we want to look at all of the visits around this point
in the sky, at high res. 

In [ ]:
# Choose a resolution
nside = 1024 * 4
# Find the healpixels which land within X of the test center location
hpid = np.arange(hp.nside2npix(nside))
hra, hdec = hp.pix2ang(nside, hpid, lonlat=True)
ang = rsUtils.angular_separation(hra, hdec, test_ra, test_dec)
# How big do we want to make the map? I don't know what a patch looks like
rad = 2
in_patch = np.where(ang <= rad)[0]

s = maf.HealpixSubsetSlicer(nside=nside, hpid=in_patch, use_cache=False)
m = maf.CountMetric("observationStartMJD")
constraint = 'band == "r" and night<365/2.0'
b = maf.MetricBundle(m, s, constraint, run_name=run_name)
g2 = maf.MetricBundleGroup({"0": b}, opsdb_fname, data_dir)
g2.run_all()

In [ ]:
plotDict = {"visufunc": hp.gnomview, "rot": (test_ra, test_dec, 0), "xsize": 200, "figsize": (8, 8)}
b.set_plot_dict(plotDict)
b.plot()

---

### Let's also look at the visits in a single night

In [ ]:
# There are lots of ways to get information out of the database when you
# just want to know "where did we look in one night" .. a simple sql query is fine
import sqlite3

# cols = ['observationStartMJD', 'fieldRA', 'fieldDec', 'rotTelPos', 'rotSkyPos', 'band', 'night', 'note']
cols = [
    "observationStartMJD",
    "fieldRA",
    "fieldDec",
    "rotTelPos",
    "rotSkyPos",
    "band",
    "night",
    "scheduler_note",
]
query = " ".join(["select", " ".join([f"{c}," for c in cols])[:-1], "from observations where night < 200"])
print(query)
conn = sqlite3.connect(opsdb_fname)
night_visits = pd.read_sql(query, conn)

conn.close()
print("nvisits total", len(night_visits))
night_visits.head()

In [ ]:
filters = night_visits["band"].unique()
filters

In [ ]:
# locations on sky (not pretty because not projected)
filter_rgb_map = {
    "u": (74 / 256, 125 / 256, 179 / 256),
    "g": (104 / 256, 173 / 256, 87 / 256),
    "r": (238 / 256, 134 / 256, 50 / 256),
    "i": (232 / 256, 135 / 256, 189 / 256),
    "z": (209 / 256, 53 / 256, 43 / 256),
    "y": (142 / 256, 82 / 256, 159 / 256),
}


def setcolor(x):
    x["color"] = filter_rgb_map[x["band"]]
    return x


night_visits = night_visits.apply(setcolor, axis=1)

fig = plt.figure(figsize=(16, 6))
plt.scatter(night_visits["fieldRA"], night_visits["fieldDec"], c=night_visits["color"], alpha=0.3)
plt.xlabel("fieldRA")
plt.ylabel("fieldDec")
plt.title("bands-color")
plt.colorbar()
plt.show()

In [ ]:
# put rotation angles back into -90 to 90
rotAngle = np.where(
    night_visits["rotTelPos"] > 180, night_visits["rotTelPos"] % 180 - 180, night_visits["rotTelPos"]
)
night_visits["rotAngle"] = rotAngle
n, b, p = plt.hist(rotAngle, bins=50)
plt.xlabel("Rotation angle (deg)")
plt.show()

In [ ]:
changes = np.diff(rotAngle)
changes = np.concatenate([[0], changes])

# Add changes for filter change
filterchange = np.where(night_visits["band"][:-1].values != night_visits["band"][1:].values)[0]
# Changes happen between filterchange and filterchange + 1
# replace the rotation angle changes at those points with change from previous - 0 - next
changes[filterchange + 1] = (
    night_visits["rotAngle"][filterchange].values + night_visits["rotAngle"][filterchange + 1].values
)

night_visits["rotChanges"] = changes
n, b, p = plt.hist(changes, bins=50)
plt.xlabel("Rotation Angle CHANGE (deg)")
plt.show()

In [ ]:
plt.figure(figsize=(16, 6))
plt.scatter(night_visits["fieldRA"], night_visits["fieldDec"], c=night_visits["rotAngle"], alpha=0.5)
plt.colorbar()
plt.xlabel("fieldRA")
plt.ylabel("fieldDec")
plt.title("rotAngle")
plt.show()
plt.figure(figsize=(16, 6))
plt.scatter(
    night_visits["fieldRA"],
    night_visits["fieldDec"],
    c=night_visits["rotChanges"],
    alpha=0.4,
    cmap="RdBu",
    vmin=-20,
    vmax=50,
)
plt.colorbar()
plt.xlabel("fieldRA")
plt.ylabel("fieldDec")
plt.title("rotChanges")
plt.show()